# SciQLop - Virtual Products

Virtual products are user-defined data sources backed by a Python callback.
SciQLop calls your function on demand as you navigate time, just like built-in data products.

This example shows three ways to create virtual products, from simplest to most flexible.

## 1. Quickest: the `%%vp` cell magic (recommended)

Write a function with a return type annotation and the magic handles everything.
Re-executing the cell hot-reloads the callback — open plots update automatically.

In [ ]:
%%vp --path "examples/cosine"
import numpy as np

def cosine(start: float, stop: float) -> Scalar["cos"]:
    x = np.arange(start, stop, 5.0)
    return x, np.cos(x / 100)

In [ ]:
from datetime import datetime, timedelta
from SciQLop.user_api.plot import create_plot_panel
from SciQLop.user_api import TimeRange

p = create_plot_panel()
p.time_range = TimeRange(datetime.now().timestamp(), (datetime.now() + timedelta(seconds=100)).timestamp())
p.plot("examples/cosine")

## 2. Convenience classes

`VirtualScalar`, `VirtualVector`, `VirtualMultiComponent`, and `VirtualSpectrogram`
provide a concise programmatic API without needing the enum.

In [ ]:
import numpy as np
from datetime import datetime, timedelta
from SciQLop.user_api.virtual_products import VirtualScalar, VirtualVector

def my_scalar(start: datetime, stop: datetime):
    x = np.arange(round(start.timestamp() / 5) * 5, round(stop.timestamp() / 5) * 5, 5, dtype=np.float64)
    return x, np.cos(x / 100)

scalar_vp = VirtualScalar("examples/my_scalar", my_scalar, label="cos", cachable=True)

In [ ]:
def my_vector(start: datetime, stop: datetime):
    t = np.arange(round(start.timestamp() / 5) * 5, round(stop.timestamp() / 5) * 5, 5, dtype=np.float64)
    return t, np.column_stack([np.cos(t / 200), np.sin(t / 200), np.zeros_like(t)])

vector_vp = VirtualVector("examples/my_vector", my_vector, labels=["Vx", "Vy", "Vz"], cachable=True)

In [ ]:
p2 = create_plot_panel()
p2.time_range = TimeRange(datetime.now().timestamp(), (datetime.now() + timedelta(seconds=200)).timestamp())
p2.plot(scalar_vp)
p2.plot(vector_vp)

## 3. Full control: `create_virtual_product`

The generic factory function accepts all product types via the `VirtualProductType` enum.
Use this when you need maximum flexibility.

In [ ]:
from SciQLop.user_api.virtual_products import create_virtual_product, VirtualProductType

def my_product_gen(start: datetime, stop: datetime):
    x = np.arange(round(start.timestamp() / 5) * 5, round(stop.timestamp() / 5) * 5, 5, dtype=np.float64)
    return x, np.cos(x / 100)

my_product = create_virtual_product(
    path="examples/my_product",
    callback=my_product_gen,
    product_type=VirtualProductType.Scalar,
    labels=["my_product"],
    cachable=True,
    debug=False
)

## Plotting

All virtual products can be plotted by passing the object or its path string.
You can also drag and drop from the product tree in the GUI.

In [ ]:
p3 = create_plot_panel()
p3.time_range = TimeRange(datetime.now().timestamp(), (datetime.now() + timedelta(seconds=100)).timestamp())

p3.plot(my_product)  # by object
# p3.plot("examples/my_product")  # by path string — equivalent